# Mozambique VACS 2019 — PUD exploration

Single file **`MOZAMBIQUE_VACS_2019_PUD.dta`** in **`data/raw/Mozambique Stata/`**. **Encoding:** use **`latin1`** with `pyreadstat` (UTF-8 may fail on string columns).

The public PUD **does not include** a dedicated respondent ID (`VACS_ID` / `PUD_ID`) or a **household** column. The notebook **states that explicitly** and builds a **deterministic composite UID** (for analysis only), following the same transparency standard as other country notebooks.

**Flow:** §1 Load → §2 column list & EDA → §3 samples & slot summaries → §4 harmonized TSV (`variable_male` / `variable_female` identical for this single file).

PDF: **`MOZAMBIQUE_VACS_2019_DataUserGuide.pdf`**.


In [1]:
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COUNTRY_DIR = ROOT / "data" / "raw" / "Mozambique Stata"
PUD_PATH = COUNTRY_DIR / "MOZAMBIQUE_VACS_2019_PUD.dta"
READ_KW = {"encoding": "latin1"}

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

`read_dta(..., **READ_KW)` with **`latin1`**.


In [2]:
if not PUD_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {PUD_PATH}")

df, meta = pyreadstat.read_dta(PUD_PATH, **READ_KW)
print(f"File: {PUD_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")

df = df.copy()

if "Sex" in df.columns:
    print("Sex (check codebook for 1=male / 2=female) :")
    display(df["Sex"].value_counts(dropna=False).sort_index())

# --- No single respondent ID in PUD: build analysis-only composite ---
# Sort keys break ties within CLUSTER so CLUSTER + within-cluster rank is unique in this file.
_sort_keys = [c for c in ["CLUSTER", "Sex", "NTOT", "H3", "H1_1", "SampleWeight", "PROV", "Strata"] if c in df.columns]
_df = df.copy()
for c in _sort_keys:
    if _df[c].isna().any() and pd.api.types.is_numeric_dtype(_df[c]):
        _df[c] = _df[c].fillna(_df[c].median())
df = _df.sort_values(_sort_keys).reset_index(drop=True)
df["RESP_UID"] = df["CLUSTER"].astype(int).astype(str) + "_" + df.groupby("CLUSTER").cumcount().astype(str)

assert df["RESP_UID"].nunique() == len(df), "Composite key not unique—update _sort_keys"

print(f"Max respondents per CLUSTER: {int(df.groupby('CLUSTER').size().max())}")
print(f"RESP_UID unique / rows: {df['RESP_UID'].nunique():,} / {len(df):,}")
_syn = [c for c in ["RESP_UID", "HH_COMPOSITE"] if c in df.columns]
print(f"Duplicate rows (excluding composites): {int(df.drop(columns=_syn, errors='ignore').duplicated().sum())}")

# No standalone household ID: operational household row = one respondent row (VACS design).
df["HH_COMPOSITE"] = df["RESP_UID"]

df.head(4)


File: /Users/starsrain/research_side_projects_ipv/data/raw/Mozambique Stata/MOZAMBIQUE_VACS_2019_PUD.dta
Rows × columns: 3,008 × 568
Sex (check codebook for 1=male / 2=female) :


Sex
1.0     879
2.0    2129
Name: count, dtype: int64

Max respondents per CLUSTER: 18
RESP_UID unique / rows: 3,008 / 3,008
Duplicate rows (excluding composites): 0


,PROV,NTOT,HHQ_LAN,H1_1,H2,H3,H4,H4A,H4B,H4C,...,PV4LWHR,PV4FWHR,PVSKHLP,PVSRHLP,SV2WHR,SV4HR,WORTH,CLUSTER,RESP_UID,HH_COMPOSITE
0,1.0,3.0,1.0,2.0,1.0,17.0,5.0,50.0,10.0,200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,20001.0,20001_0,20001_0
1,1.0,6.0,1.0,2.0,2.0,23.0,5.0,100.0,30.0,300.0,...,NaN,NaN,1.0,NaN,NaN,NaN,1.0,20001.0,20001_1,20001_1
2,1.0,6.0,1.0,2.0,2.0,24.0,5.0,20.0,15.0,200.0,...,NaN,NaN,1.0,NaN,NaN,NaN,1.0,20001.0,20001_2,20001_2
3,1.0,9.0,1.0,1.0,1.0,39.0,5.0,10.0,10.0,600.0,...,NaN,NaN,1.0,NaN,NaN,NaN,1.0,20001.0,20001_3,20001_3


In [7]:
df['RESP_UID']

0        20001_0
1        20001_1
2        20001_2
3        20001_3
4        20001_4
          ...   
3003     20377_6
3004     20377_7
3005     20377_8
3006     20377_9
3007    20377_10
Name: RESP_UID, Length: 3008, dtype: str

In [ ]:
df['PROV']
""" 
df['CLUSTER']
df['Sex']
df['NTOT']
df['H3']
df['H1_1'] """

0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
3003    8.0
3004    8.0
3005    8.0
3006    8.0
3007    8.0
Name: PROV, Length: 3008, dtype: float64

In [12]:
df['CLUSTER']


0       20001.0
1       20001.0
2       20001.0
3       20001.0
4       20001.0
         ...   
3003    20377.0
3004    20377.0
3005    20377.0
3006    20377.0
3007    20377.0
Name: CLUSTER, Length: 3008, dtype: float64

## 2. Column list & quick EDA


In [3]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


Variables: 570  |  Observations: 3,008


,column,stata_label,dtype,missing_n,missing_pct
PROV,PROV,Province,float64,0,0.00
NTOT,NTOT,How many people are living in this household?,float64,0,0.00
HHQ_LAN,HHQ_LAN,Which language are you using for the Head of t...,float64,11,0.37
H1_1,H1_1,INTERVIEWER: PLEASE RECORD IF YOU ARE INTERVIE...,float64,0,0.00
H2,H2,H2. RECORD THE SEX OF THE HEAD OF HOUSEHOLD:,float64,11,0.37
H3,H3,H3. How old are you?,float64,0,0.00
H4,H4,H4. What is the main source of drinking water ...,float64,11,0.37
H4A,H4A,H4A. How far is it from your home to the place...,float64,576,19.15
H4B,H4B,H4B. How long does it take to walk there and b...,float64,576,19.15
H4C,H4C,H4C. What is the distance (in meters) you trav...,float64,11,0.37


,column,stata_label,dtype,missing_n,missing_pct
0,PVknewplace,,float64,3008,100.00
1,H63_4,Injury type person #4,float64,3008,100.00
2,H63_5,Injury type person #5,float64,3008,100.00
3,HVARVNR,,float64,3007,99.97
4,H62_4,Injury type who killed person #4,float64,3007,99.97
5,H62_5,Injury type who killed person #5,float64,3007,99.97
6,H63_3,Injury type person #3,float64,3007,99.97
7,HVARVCR,,float64,3006,99.93
8,H62_3,Injury type who killed person #3,float64,3004,99.87
9,SV5WHR,,float64,3003,99.83


<class 'pandas.DataFrame'>
RangeIndex: 3008 entries, 0 to 3007
Columns: 570 entries, PROV to HH_COMPOSITE
dtypes: float64(538), str(32)
memory usage: 13.1 MB


In [6]:
var_table['RESP_UID']

KeyError: 'RESP_UID'

## 3. Further EDA and exploration

### Raw row samples


In [5]:
pd.set_option("display.max_columns", 42)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [c for c in [
    "RESP_UID", "HH_COMPOSITE", "CLUSTER", "PROV", "Strata", "Sex",
    "SampleWeight", "HIVWeight", "HIVPSWeight", "NTOT", "H3",
] if c in df.columns]
sub = df[_core]
display(sub.head(8))
display(sub.sample(6, random_state=0))


,RESP_UID,HH_COMPOSITE,CLUSTER,PROV,Strata,Sex,SampleWeight,HIVWeight,HIVPSWeight,NTOT,H3
0,20001_0,20001_0,20001.0,1.0,2125.0,2.0,5893.464809,8715.515,1.022146,3.0,17.0
1,20001_1,20001_1,20001.0,1.0,2125.0,2.0,9847.103700,NaN,1.022146,6.0,23.0
2,20001_2,20001_2,20001.0,1.0,2125.0,2.0,7524.443577,8715.515,1.022146,6.0,24.0
3,20001_3,20001_3,20001.0,1.0,2125.0,2.0,9847.103700,NaN,1.022146,9.0,39.0
4,20001_4,20001_4,20001.0,1.0,2125.0,2.0,7524.443577,8715.515,1.022146,10.0,39.0
5,20002_0,20002_0,20002.0,1.0,2124.0,2.0,2831.332164,8715.515,1.022146,2.0,18.0
6,20002_1,20002_1,20002.0,1.0,2124.0,2.0,2831.332164,NaN,1.022146,2.0,19.0
7,20002_2,20002_2,20002.0,1.0,2124.0,2.0,1977.819931,NaN,1.022146,2.0,23.0


,RESP_UID,HH_COMPOSITE,CLUSTER,PROV,Strata,Sex,SampleWeight,HIVWeight,HIVPSWeight,NTOT,H3
1830,20224_6,20224_6,20224.0,9.0,2113.0,2.0,232.589311,466.332465,1.022146,9.0,17.0
361,20053_2,20053_2,20053.0,4.0,2128.0,2.0,767.112266,NaN,1.022146,3.0,17.0
1191,20147_2,20147_2,20147.0,4.0,2128.0,2.0,799.279191,1047.480983,1.022146,4.0,70.0
2244,20272_4,20272_4,20272.0,9.0,2113.0,2.0,151.871804,345.729242,1.022146,8.0,39.0
2533,20311_1,20311_1,20311.0,11.0,2119.0,2.0,2121.497696,6486.536588,1.022146,3.0,54.0
2722,20338_5,20338_5,20338.0,3.0,1123.0,1.0,3128.900918,4913.440567,0.969268,5.0,27.0


### Slot summaries

Widths, optional **style**, and **Male/Female word** heuristic (not single-letter M/F codes).


In [4]:
L = meta.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (str(L.get(c) or ""))[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Respondent (no official ID — composite)", ["RESP_UID"], "see §1: CLUSTER + within-CLUSTER rank after deterministic sort")
slot_summary("2. Household (no hh column — composite)", ["HH_COMPOSITE"], "alias of RESP_UID; one row ≈ one household (VACS adolescent)")
slot_summary("3. Province", ["PROV"], "")
slot_summary("4. Strata (numeric codes)", ["Strata"], "e.g. 1111–2129 pattern; pair with codebook / guide")
slot_summary("5. Cluster / PSU (EA codes)", ["CLUSTER"], "5-digit codes in 20001–20377 range here")
slot_summary("6. Sex", ["Sex"], "")
slot_summary("7. Weights", ["SampleWeight", "HIVWeight", "HIVPSWeight"], "")
slot_summary("8. NTOT (HH size, not an ID)", ["NTOT"], "")



1. Respondent (no official ID — composite)
  RESP_UID | 
    7-8 chars (string); no male/female text (heuristic); dtype=str; n_distinct=3008; missing=0
   see §1: CLUSTER + within-CLUSTER rank after deterministic sort

2. Household (no hh column — composite)
  HH_COMPOSITE | 
    7-8 chars (string); no male/female text (heuristic); dtype=str; n_distinct=3008; missing=0
   alias of RESP_UID; one row ≈ one household (VACS adolescent)

3. Province
  PROV | Province
    1-2 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1.0/11.0; n_distinct=10; missing=0

4. Strata (numeric codes)
  Strata | Strata
    4 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1111.0/2129.0; n_distinct=38; missing=0
   e.g. 1111–2129 pattern; pair with codebook / guide

5. Cluster / PSU (EA codes)
  CLUSTER | 
    5 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=20001.0/20377.0; n_distinct=377; missing=0
   5-digit codes in 20001

## 4. Harmonized codebook slots (Mozambique 2019)

**Single combined PUD** — **`variable_male`** and **`variable_female`** repeat the same names; use **`Sex`** and **notes**.

**Source:** `data/raw/Mozambique Stata/MOZAMBIQUE_VACS_2019_PUD.dta`. Read with **`latin1`**.

### When there is no single variable

1. **Respondent ID:** There is **no** producer respondent ID in this extract. **Suggested composite (analysis only):** sort rows by **`CLUSTER`, `Sex`, `NTOT`, `H3`, `H1_1`, `SampleWeight`, `PROV`, `Strata`** (with brief median fill for missing numeric sort keys), then  
   **`RESP_UID` = `str(int(CLUSTER)) + "_" + str(rank_within_CLUSTER)`**  
   This is **unique in this file** but **not** a released survey ID—document that in Excel **notes**.

2. **Household ID:** There is **no** `hh` / household key column. Under standard VACS (one sampled adolescent per household), **each row is one household**; use **`HH_COMPOSITE = RESP_UID`** as an **operational** household key, or document **“no separate household variable—use composite above.”**

```
slot	variable_male	variable_female	type_and_width	notes
Respondent ID	(see §1 composite) RESP_UID	(see §1 composite) RESP_UID	str; pattern `NNNNN_rank` (e.g. 20001_0)	**No official ID in PUD**; composite is sort-dependent—keep `_sort_keys` fixed if reproducing
Household ID	(see §1) HH_COMPOSITE	(see §1) HH_COMPOSITE	same as RESP_UID	**No `hh` column**; one row per household interview; CLUSTER alone has many HHs per EA
Geo level 1	PROV	PROV	1–2 digits (integer); 10 province codes in data (1–11 with **7** absent)	Stata label: Province
Geo level 2	Strata	Strata	4-digit integer codes (e.g. 1111–2129)	Selection / design strata; interpret with codebook/ guide (often encodes geography × design cells)
Geo level 3	—	—	—	Not separately released beyond Strata pattern here
Cluster / PSU	CLUSTER	CLUSTER	float → **5-digit** integer string (20001–20377)	PSU / EA; multiple household interviews per CLUSTER in file
Sex	Sex	Sex	1 / 2 (integer)	Confirm 1=male / 2=female in codebook
Weight	SampleWeight; HIVWeight; HIVPSWeight	SampleWeight; HIVWeight; HIVPSWeight	float	`SampleWeight` 1 missing row in extract; `HIVWeight` many missing (HIV module)
Interview date	—	—	—	**No** person-level interview date fields spotted in initial pass (only violence “12 month” month fields, etc.)—confirm in codebook / guide; document fieldwork window from PDF if needed
```

**Excel:** Spell out **missing official IDs** and **composites** in **notes** so the ID / Geographic tabs stay auditable.
